In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from textblob import TextBlob
from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, MaxAbsScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

BASE_DIR = Path(".")
TWEETS_PATH = BASE_DIR / "tweets.txt"
EMOJI_PATH = BASE_DIR / "emoji.txt"
OUTPUT_DIR = BASE_DIR / "xgb_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#\w+")
RT_PATTERN = re.compile(r"^RT\s+")
NON_ALPHA_PATTERN = re.compile(r"[^a-z\s]")
MULTISPACE_PATTERN = re.compile(r"\s+")

BRANDS = [
    "starbucks",
    "mcdonalds",
    "subway",
    "dominos",
    "dunkindonuts",
    "pizzahut",
    "walmart",
    "burger king",
    "chipotle",
    "five guys"
]

CRAVING_WORDS = {
    "want", "need", "craving", "crave", "hungry", "starving", "thirsty",
    "delicious", "yummy", "taste", "tasty", "frappe", "latte", "coffee",
    "burger", "pizza", "fries", "cookie", "sandwich", "tea", "drink"
}

POSITIVE_HINTS = {
    "love", "great", "good", "best", "happy", "amazing", "perfect", "lit",
    "favorite", "excited", "obsessed", "yum", "yummy", "delicious", "fresh"
}

NEGATIVE_HINTS = {
    "hate", "sad", "bad", "wrong", "nasty", "disgusting", "burnt", "late",
    "slow", "ruined", "broken", "annoyed", "cry", "tired", "mad"
}


def load_data(tweets_path: Path = TWEETS_PATH, emoji_path: Path = EMOJI_PATH) -> pd.DataFrame:
    tweets = tweets_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    labels = emoji_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    if len(tweets) != len(labels):
        raise ValueError(f"Row mismatch: {len(tweets)=}, {len(labels)=}")
    return pd.DataFrame({"text": tweets, "label": labels})


def clean_text(text: str) -> str:
    text = str(text).lower()
    text = RT_PATTERN.sub(" ", text)
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(lambda m: f" {m.group()[1:]} ", text)
    text = NON_ALPHA_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text


def sentiment_scores(text: str) -> pd.Series:
    blob = TextBlob(text).sentiment
    polarity = float(blob.polarity)
    subjectivity = float(blob.subjectivity)
    return pd.Series(
        {
            "sent_neg": max(-polarity, 0.0),
            "sent_neu": max(0.0, 1.0 - abs(polarity)),
            "sent_pos": max(polarity, 0.0),
            "sent_compound": polarity,
            "sent_subjectivity": subjectivity,
        }
    )


def count_matches(text: str, terms: set[str]) -> int:
    tokens = text.split()
    return sum(token in terms for token in tokens)


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["clean_text"] = df["text"].map(clean_text)

    df["char_count"] = df["text"].str.len()
    df["word_count"] = df["clean_text"].str.split().str.len().fillna(0)
    df["has_link"] = df["text"].str.contains(r"http\S+|www\.\S+", regex=True).astype(int)
    df["has_mention"] = df["text"].str.contains(r"@\w+", regex=True).astype(int)
    df["has_hashtag"] = df["text"].str.contains(r"#\w+", regex=True).astype(int)
    df["is_retweet"] = df["text"].str.startswith("RT ").astype(int)
    df["exclamation_count"] = df["text"].str.count("!")
    df["question_count"] = df["text"].str.count(r"\?")
    df["digit_count"] = df["text"].str.count(r"\d")
    df["upper_ratio"] = df["text"].apply(
        lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1)
    )
    df["elongated_count"] = df["text"].str.count(r"(.)\1{2,}")

    df["craving_word_count"] = df["clean_text"].apply(lambda x: count_matches(x, CRAVING_WORDS))
    df["positive_hint_count"] = df["clean_text"].apply(lambda x: count_matches(x, POSITIVE_HINTS))
    df["negative_hint_count"] = df["clean_text"].apply(lambda x: count_matches(x, NEGATIVE_HINTS))

    sentiment_df = df["clean_text"].apply(sentiment_scores)
    df = pd.concat([df, sentiment_df], axis=1)

    for brand in BRANDS:
        col = "brand_" + brand.replace(" ", "_")
        if " " in brand:
            df[col] = df["text"].str.lower().str.contains(re.escape(brand), regex=True).astype(int)
        else:
            df[col] = df["text"].str.lower().str.contains(r"\b" + re.escape(brand) + r"\b", regex=True).astype(int)

    df["brand_mention_total"] = df[[c for c in df.columns if c.startswith("brand_")]].sum(axis=1)
    return df


def build_matrices(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    text_col: str,
    numeric_cols: list[str],
    max_word_features: int = 40000,
    max_char_features: int = 25000,
    svd_components: int = 300,
):
    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
        max_features=max_word_features,
    )

    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=3,
        max_df=0.98,
        sublinear_tf=True,
        max_features=max_char_features,
    )

    X_train_word = word_vectorizer.fit_transform(train_df[text_col])
    X_test_word = word_vectorizer.transform(test_df[text_col])

    X_train_char = char_vectorizer.fit_transform(train_df[text_col])
    X_test_char = char_vectorizer.transform(test_df[text_col])

    text_train = hstack([X_train_word, X_train_char], format="csr")
    text_test = hstack([X_test_word, X_test_char], format="csr")

    svd = TruncatedSVD(n_components=svd_components, random_state=42)
    X_train_text_dense = svd.fit_transform(text_train)
    X_test_text_dense = svd.transform(text_test)

    scaler = MaxAbsScaler()
    X_train_num = scaler.fit_transform(train_df[numeric_cols])
    X_test_num = scaler.transform(test_df[numeric_cols])

    X_train_final = np.hstack([X_train_text_dense, np.asarray(X_train_num)])
    X_test_final = np.hstack([X_test_text_dense, np.asarray(X_test_num)])

    return X_train_final, X_test_final, word_vectorizer, char_vectorizer, svd, scaler


def export_presentation_csvs(df: pd.DataFrame, class_metrics: pd.DataFrame, cm_df: pd.DataFrame, accuracy: float) -> None:
    brand_cols = [c for c in df.columns if c.startswith("brand_") and c != "brand_mention_total"]

    brand_sentiment_rows = []
    for col in brand_cols:
        brand_df = df[df[col] == 1]
        if len(brand_df) == 0:
            continue
        brand_sentiment_rows.append(
            {
                "brand": col.replace("brand_", "").replace("_", " ").title(),
                "mention_count": int(len(brand_df)),
                "avg_sent_compound": float(brand_df["sent_compound"].mean()),
                "avg_sent_pos": float(brand_df["sent_pos"].mean()),
                "avg_sent_neg": float(brand_df["sent_neg"].mean()),
                "avg_craving_word_count": float(brand_df["craving_word_count"].mean()),
            }
        )

    brand_sentiment_df = pd.DataFrame(brand_sentiment_rows).sort_values("mention_count", ascending=False)
    brand_sentiment_df.to_csv(OUTPUT_DIR / "brand_sentiment_scatter.csv", index=False)

    heatmap_rows = []
    for col in brand_cols:
        brand_name = col.replace("brand_", "").replace("_", " ").title()
        subset = df[df[col] == 1]
        if len(subset) == 0:
            continue
        top_emotions = subset["label"].value_counts(normalize=True).head(5)
        for emotion, share in top_emotions.items():
            heatmap_rows.append({"brand": brand_name, "emotion": emotion, "share": float(share)})
    pd.DataFrame(heatmap_rows).to_csv(OUTPUT_DIR / "brand_emotion_heatmap.csv", index=False)

    trend_df = df.copy()
    trend_df["chunk"] = pd.qcut(np.arange(len(trend_df)), q=20, duplicates="drop")
    trend_csv = (
        trend_df.groupby("chunk", observed=False)
        .agg(
            avg_craving_score=("craving_word_count", "mean"),
            avg_sent_compound=("sent_compound", "mean"),
            avg_brand_mentions=("brand_mention_total", "mean"),
            tweet_count=("text", "size"),
        )
        .reset_index()
    )
    trend_csv["chunk_id"] = np.arange(1, len(trend_csv) + 1)
    trend_csv.drop(columns=["chunk"]).to_csv(OUTPUT_DIR / "craving_signal_timeline.csv", index=False)

    class_metrics.to_csv(OUTPUT_DIR / "xgb_classification_report.csv", index=False)
    cm_df.to_csv(OUTPUT_DIR / "xgb_confusion_matrix.csv", index=False)

    summary_df = pd.DataFrame(
        [
            {"metric": "dataset_rows", "value": int(len(df))},
            {"metric": "num_classes", "value": int(df["label"].nunique())},
            {"metric": "target_accuracy_threshold", "value": 0.70},
            {"metric": "xgboost_test_accuracy", "value": float(accuracy)},
            {"metric": "avg_sent_compound", "value": float(df["sent_compound"].mean())},
            {"metric": "avg_craving_word_count", "value": float(df["craving_word_count"].mean())},
        ]
    )
    summary_df.to_csv(OUTPUT_DIR / "presentation_kpis.csv", index=False)


def run_xgboost_pipeline(sample_size: int | None = 120000):
    df = load_data()
    df = add_features(df)

    if sample_size is not None and sample_size < len(df):
        df, _ = train_test_split(df, train_size=sample_size, random_state=42, stratify=df["label"])

    numeric_cols = [
        "char_count",
        "word_count",
        "has_link",
        "has_mention",
        "has_hashtag",
        "is_retweet",
        "exclamation_count",
        "question_count",
        "digit_count",
        "upper_ratio",
        "elongated_count",
        "craving_word_count",
        "positive_hint_count",
        "negative_hint_count",
        "sent_neg",
        "sent_neu",
        "sent_pos",
        "sent_compound",
        "sent_subjectivity",
        "brand_mention_total",
    ] + [c for c in df.columns if c.startswith("brand_") and c != "brand_mention_total"]

    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

    X_train, X_test, _, _, _, _ = build_matrices(
        train_df=train_df,
        test_df=test_df,
        text_col="clean_text",
        numeric_cols=numeric_cols,
        max_word_features=40000,
        max_char_features=25000,
        svd_components=300,
    )

    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(train_df["label"])
    y_test = label_encoder.transform(test_df["label"])

    base_model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=4,
    )

    search_space = {
        "n_estimators": [200, 300, 450],
        "learning_rate": [0.05, 0.08, 0.12],
        "max_depth": [5, 6, 8],
        "min_child_weight": [1, 2, 4],
        "subsample": [0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0],
        "gamma": [0.0, 0.2, 0.5],
        "reg_lambda": [1.0, 2.0, 5.0],
    }

    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=search_space,
        n_iter=12,
        scoring="accuracy",
        cv=3,
        verbose=2,
        random_state=42,
        n_jobs=1,
    )

    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    predictions = best_model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    report = classification_report(
        y_test,
        predictions,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    class_metrics = (
        pd.DataFrame(report)
        .transpose()
        .reset_index()
        .rename(columns={"index": "label"})
    )

    cm = confusion_matrix(y_test, predictions)
    cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)

    export_presentation_csvs(df, class_metrics, cm_df, accuracy)

    print("\n=== Best XGBoost Params ===")
    print(search.best_params_)

    print("\n=== XGBoost Accuracy ===")
    print(round(accuracy, 4))

    print("\n=== Top Rows: Presentation KPIs ===")
    print(pd.read_csv(OUTPUT_DIR / "presentation_kpis.csv"))

    print("\n=== Top Rows: Brand Sentiment Scatter CSV ===")
    print(pd.read_csv(OUTPUT_DIR / "brand_sentiment_scatter.csv").head())

    print("\n=== Top Rows: Class Report CSV ===")
    print(pd.read_csv(OUTPUT_DIR / "xgb_classification_report.csv").head())

    make_presentation_plots()

    return {
        "accuracy": accuracy,
        "best_params": search.best_params_,
        "output_dir": str(OUTPUT_DIR),
    }


def make_presentation_plots() -> None:
    brand_sentiment = pd.read_csv(OUTPUT_DIR / "brand_sentiment_scatter.csv")
    class_report = pd.read_csv(OUTPUT_DIR / "xgb_classification_report.csv")
    timeline = pd.read_csv(OUTPUT_DIR / "craving_signal_timeline.csv")
    heatmap = pd.read_csv(OUTPUT_DIR / "brand_emotion_heatmap.csv")

    plt.figure(figsize=(10, 6))
    plt.scatter(brand_sentiment["mention_count"], brand_sentiment["avg_sent_compound"], s=brand_sentiment["avg_craving_word_count"] * 350 + 50)
    for _, row in brand_sentiment.iterrows():
        plt.annotate(row["brand"], (row["mention_count"], row["avg_sent_compound"]), fontsize=9)
    plt.xlabel("Brand Mention Count")
    plt.ylabel("Average Compound Sentiment")
    plt.title("Brand Mentions vs Sentiment")
    plt.tight_layout()
    plt.show()

    class_only = class_report[
        ~class_report["label"].isin(["accuracy", "macro avg", "weighted avg"])
    ].copy()
    class_only["f1-score"] = pd.to_numeric(class_only["f1-score"], errors="coerce")

    plt.figure(figsize=(10, 6))
    plt.plot(class_only["label"], class_only["f1-score"], marker="o")
    plt.xticks(rotation=45)
    plt.xlabel("Emoji Class")
    plt.ylabel("F1 Score")
    plt.title("Per-Class F1 Score Line Chart")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 6))
    plt.plot(timeline["chunk_id"], timeline["avg_craving_score"], marker="o", label="Avg Craving Score")
    plt.plot(timeline["chunk_id"], timeline["avg_sent_compound"], marker="s", label="Avg Sentiment")
    plt.xlabel("Tweet Chunk")
    plt.ylabel("Average Signal")
    plt.title("Craving and Sentiment Trend Over Time Chunks")
    plt.legend()
    plt.tight_layout()
    plt.show()

    pivot_heatmap = heatmap.pivot(index="brand", columns="emotion", values="share").fillna(0)
    plt.figure(figsize=(11, 6))
    plt.imshow(pivot_heatmap.values, aspect="auto")
    plt.colorbar(label="Emotion Share")
    plt.xticks(range(len(pivot_heatmap.columns)), pivot_heatmap.columns, rotation=45, ha="right")
    plt.yticks(range(len(pivot_heatmap.index)), pivot_heatmap.index)
    plt.title("Brand Emotion Heatmap")
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    run_xgboost_pipeline(sample_size=120000)


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=62b91583-1382-466d-9446-3dc5e725c312' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>